# Composite Fundamental Screener — Colab Quickstart

Runs the full pipeline end to end on a **50-ticker subset** of the S&P 500 (set `FULL_UNIVERSE = True` below to scale to all ~500 names — expect ~30 min of rate-limited EDGAR ingestion). `--universe russell3000` swaps in the Russell 3000 the same way — the committed 300-name cap mostly overlaps the S&P 500 (286/300 names), so treat it as a pipeline consistency check rather than a broader-universe test; see the README caveat.

Steps: clone → install → PIT ingest → scores/composite/backtest → validation → key charts inline — then three short extensions: **India** (a second market, no API key needed), **survivorship correction** (re-running with a point-in-time constituent filter), and the **rolling out-of-sample DSR chart** compared across markets.

Korea (DART) needs a free registered API key (`DART_API_KEY`) that most Colab sessions won't have set — its cell detects the missing key and skips gracefully rather than failing.

In [ ]:
# If running in Colab, clone the repo first (skip locally):
import os
if not os.path.exists('config.py'):
    # Replace with your repo URL, or upload the project folder to Colab.
    !git clone https://github.com/YOUR_USER/fundamental-screener.git
    %cd fundamental-screener
%pip install -q -e ./pit_fundamentals
%pip install -q -r requirements.txt

In [ ]:
FULL_UNIVERSE = False  # flip to True for the full S&P 500

import os
os.environ.setdefault('SEC_USER_AGENT', 'fundamental-screener colab REPLACE_ME@example.com')

import config
from dataclasses import replace
from screener.universes import SP500

if not FULL_UNIVERSE:
    config.MAX_TICKERS = config.SMALL_UNIVERSE_SIZE  # 50 tickers keeps Colab runtime reasonable
    # Deciles need MIN_NAMES_PER_BUCKET (5) x 10 = 50 FULLY-SCORED names in
    # every cross-section, which a 50-raw-ticker sample never reliably
    # clears once tag exclusions thin it out. Quintiles need only 25 — the
    # same reasoning KOSPI's smaller universe already uses, applied here to
    # the demo subset rather than a hand-picked exception.
    demo_universe = replace(SP500, n_buckets=5)
else:
    demo_universe = SP500

In [ ]:
# 1) Point-in-time ingestion from SEC EDGAR (resumable; disk-cached)
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
from screener.universe import get_sp500_constituents
from pit_fundamentals.ingest import run_ingest

tickers = get_sp500_constituents()['ticker'].tolist()
run_ingest(tickers, db_path=str(config.DB_PATH))

In [ ]:
# 2) Scores -> sector-neutral z -> LASSO composite -> decile backtest
from screener.backtest import run_pipeline
out = run_pipeline(demo_universe)
panel, dec_rets, coefs = out['panel'], out['decile_returns'], out['coefs']
panel.tail()

In [ ]:
# 3) Statistical validation: Newey-West + Deflated Sharpe Ratio
from screener.validation import build_summary, rolling_spread
import config as cfg
summary = build_summary(panel, dec_rets, n_buckets=demo_universe.n_buckets)
summary.to_csv(cfg.VALIDATION_SUMMARY_PATH)
roll = rolling_spread(dec_rets['spread'].dropna())
roll.to_parquet(cfg.ROLLING_SPREAD_PATH)
summary.round(3)

In [ ]:
# 4) Key dashboard charts, rendered inline
from dashboard.app import fig_decile_cumret, fig_rolling, fig_sector_heatmap, fig_f_scatter
fig_decile_cumret(dec_rets, demo_universe.n_buckets).show()
fig_rolling(roll, "sp500").show()
fig_sector_heatmap(panel.reset_index()).show()
fig_f_scatter(panel.reset_index()).show()

The rolling-spread chart is deliberately titled neutrally — whether it shows persistence or decay is an empirical result, reported as-is. See `FINDINGS.md` for the research memo.

## Extension 1: India — a second market, no API key

India's data landscape is genuinely worse than the US/Korea/Brazil sources: no single free source publishes both structured values and a real filing date, so this adapter *joins* two free sources — BSE's announcement API (real dissemination timestamps, but only PDF attachments) and Yahoo Finance's `.NS` statements (complete structured values, but no filing date at all). Yahoo values are gated by the BSE date of the announcement that first reported that period.

The result yields only ~3-4 usable annual fundamental updates per company — enough to rank a screen, nowhere near enough for Newey-West/Deflated-Sharpe inference — so India runs through `run_screen()` (scores + bucket returns + rolling chart) rather than `run_pipeline()` (which also fits the LASSO composite and builds the validation table). `backtestable=False` on the `INDIA` universe is what enforces this at the code level, not a manual choice each time.

In [ ]:
# India: ingest (no key needed) then run the screen — scores + bucket returns
# + rolling chart, but no LASSO fit and no validation table (see markdown above).
from pit_fundamentals.india_client import run_india_ingest
from screener.universe_in import get_in_universe
from screener.universes import INDIA
from screener.backtest import run_screen

run_india_ingest(get_in_universe(), db_path=str(INDIA.db_path))
india_panel = run_screen(INDIA)
india_panel.tail()

## Extension 2: Korea (KOSPI) — needs a free `DART_API_KEY`

DART (Korea's Financial Supervisory Service filing system) requires a registered key on every call, including the free company-code list — there is no code path that runs without one, unlike FMP's optional gap-fill key. Register at <https://opendart.fss.or.kr> ("인증키 신청/관리"; approval is typically near-instant), then `export DART_API_KEY=...` **before** starting this Colab session (or set it in the cell below) — never commit it to a file.

This cell detects a missing key and skips rather than failing, since most Colab sessions won't have one set.

In [ ]:
import os

korea_ran = False
if not os.environ.get("DART_API_KEY"):
    print("DART_API_KEY not set — skipping the Korea extension. "
          "Set it and re-run this cell to include KOSPI below.")
else:
    from pit_fundamentals.dart_kr_client import run_dart_ingest
    from screener.universe_kr import get_kr_blue_chips
    from screener.universes import KOSPI
    from screener.backtest import run_pipeline as run_pipeline_kr

    run_dart_ingest(get_kr_blue_chips(), years=[str(y) for y in range(2015, 2024)],
                     db_path=str(KOSPI.db_path))
    kr_out = run_pipeline_kr(KOSPI)
    kr_panel, kr_dec_rets = kr_out["panel"], kr_out["decile_returns"]
    korea_ran = True
    kr_panel.tail()

## Extension 3: Survivorship correction

`Universe.corrected()` returns a twin that restricts every rebalance date to names actually listed/constituent on that date, writing to separate `*_pit` outputs so the static and corrected runs can be compared side by side rather than one silently overwriting the other. For the S&P 500, the source is Wikipedia's constituent-changes table unwound backwards from today's list — on the full 503-name universe the 2012 cross-section shrinks to roughly 294; on this notebook's 50-ticker demo subset the same filter applies proportionally (`FULL_UNIVERSE = True` above reproduces the README's exact 503 -> 294 figure).

This reuses the PIT database and price cache already populated in Section 1 above — only the per-date universe filter differs, so this cell is fast even though the first EDGAR ingest was not.

In [ ]:
from screener.backtest import run_pipeline as run_pipeline_pit
from screener.validation import build_summary as build_summary_pit

sp500_pit = demo_universe.corrected()  # same bucket count as the static run above
pit_out = run_pipeline_pit(sp500_pit)
pit_panel, pit_dec_rets = pit_out["panel"], pit_out["decile_returns"]
pit_summary = build_summary_pit(pit_panel, pit_dec_rets, n_buckets=demo_universe.n_buckets)

import pandas as pd
pd.concat(
    {"static": summary["ann_return"], "survivorship-corrected": pit_summary["ann_return"]},
    axis=1,
).round(4)

## Extension 4: Rolling out-of-sample chart, compared across markets

Each window's shaded band is derived from the Deflated Sharpe Ratio, not a plain standard-error band: it marks the spread that window would need for its *own* DSR to reach 95%, given that window's volatility, empirical skew/kurtosis, and length. A line sitting inside the band would not have survived the same multiple-testing correction the summary table applies — "outside the band" and "DSR ≥ 0.95" agree on every test window by construction.

India's chart uses the same 24-month window as the others for visual consistency, even though its ~27 monthly cross-sections leave only ~3 overlapping windows — see `FINDINGS.md` for why that is reported as a shape diagnostic, not evidence of decay one way or the other. Korea's chart only renders if the DART extension above ran.

In [ ]:
import pandas as pd
from pathlib import Path

fig_rolling(roll, "sp500").show()

if Path(INDIA.rolling_path).exists():
    india_roll = pd.read_parquet(INDIA.rolling_path)
    fig_rolling(india_roll, "india", backtestable=False).show()

if korea_ran and Path(KOSPI.rolling_path).exists():
    kr_roll = pd.read_parquet(KOSPI.rolling_path)
    fig_rolling(kr_roll, "kospi").show()
elif korea_ran:
    from screener.validation import rolling_spread
    kr_roll = rolling_spread(kr_dec_rets["spread"].dropna())
    fig_rolling(kr_roll, "kospi").show()